# Tên công ty

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

BASE_URL = "https://infodoanhnghiep.com"
LIST_URL = "https://infodoanhnghiep.com/Hai-Phong/"

headers = {
    "User-Agent": "Mozilla/5.0"
}


def get_company_list(url):
    try:
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        companies = []

        items = soup.find_all("div", class_="company-item")

        if not items:
            return []

        for item in items:
            name_tag = item.find("h3")
            name = name_tag.get_text(strip=True) if name_tag else "N/A"

            address_tag = item.find("p")
            address = address_tag.get_text(strip=True) if address_tag else "N/A"
            address = address.split("\n")[0]

            link_tag = item.find("a")
            link = None
            if link_tag:
                href = link_tag.get("href")
                if href and href.startswith("http"):
                    link = href
                elif href:
                    link = BASE_URL + href

            companies.append({
                "name": name,
                "address": address,
                "link": link
            })

        return companies

    except Exception as e:
        print("❌ Lỗi page:", url, e)
        return []


def get_mst(detail_url):
    try:
        res = requests.get(detail_url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        text = soup.get_text()

        match = re.search(r"Mã số thuế[:\s]+(\d+)", text)
        return match.group(1) if match else None

    except Exception as e:
        print("❌ Lỗi MST:", e)
        return None


# ===== 🆕 CRAWL ALL PAGES =====
def get_all_companies():
    all_companies = []
    page = 1

    while True:
        if page == 1:
            url = LIST_URL
        else:
            url = f"{LIST_URL}trang-{page}/"

        print(f"🔎 Page {page}: {url}")

        companies = get_company_list(url)

        if not companies:
            print("⛔ Hết dữ liệu")
            break

        all_companies.extend(companies)

        page += 1
        time.sleep(1)  # tránh bị block

    return all_companies


# ===== MAIN =====
companies = get_all_companies()

print(f"✅ Tổng công ty: {len(companies)}")

# ===== LẤY MST  =====
for i, c in enumerate(companies):
    print(f"{i+1}/{len(companies)}: {c['name']}")

    if c["link"]:
        c["mst"] = get_mst(c["link"])
    else:
        c["mst"] = None

    time.sleep(1)

# ===== SAVE FILE =====
with open("companies_full.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=4)

print("✅ Done")

# 2. Mã số thuế


In [ ]:
!wget https://masothue.com/0801459243-cong-ty-tnhh-vn-home -O test.html

In [ ]:
import subprocess
from bs4 import BeautifulSoup
import json
import re
import unicodedata
import time
import random


# ========================
# FETCH HTML (wget)
# ========================
def fetch_html(url):
    result = subprocess.run(
        ["wget", "-qO-", url],
        capture_output=True,
        text=True
    )
    return result.stdout


# ========================
# RETRY (anti-block)
# ========================
def fetch_html_retry(url, retries=3):
    for i in range(retries):
        html = fetch_html(url)

        # check bị block
        if "Enable JavaScript and cookies to continue" not in html:
            return html

        print(f"⚠️ Bị block, retry lần {i+1}")
        time.sleep(random.uniform(2, 5))

    return html


# ========================
# SLUGIFY
# ========================
def slugify(text):
    text = text.lower()

    text = text.replace("đ", "d")

    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')

    text = re.sub(r'[^a-z0-9]+', '-', text)

    return text.strip('-')


# ========================
# BUILD URL
# ========================
def build_url(mst, name):
    slug = slugify(name)
    return f"https://masothue.com/{mst}-{slug}"


# ========================
# PARSE COMPANY
# ========================
def parse_company(mst, name):
    url = build_url(mst, name)
    html = fetch_html_retry(url)

    soup = BeautifulSoup(html, "html.parser")
    name_tag = soup.select_one("thead span.copy")
    # print("name_tag: ", name_tag)

    if not name_tag:
        print("⚠️ Slug sai → fallback MST")
        print("url old: ", url)
        url = f"https://masothue.com/{mst}"
        html = fetch_html_retry(url)


    data = {
        "mst": mst,
        "url": url,
        "name": None,
        "address": None,
        "status": None,
        "business": {
            "main": None,
            "detail": None
        }
    }


    if name_tag:
        data["name"] = name_tag.get_text(strip=True)


    rows = soup.select("tbody tr")

    for row in rows:
        cols = row.find_all("td")
        if len(cols) < 2:
            continue

        label = cols[0].get_text(strip=True)
        value_td = cols[1]


        if label == "Địa chỉ":
            data["address"] = value_td.get_text(" ", strip=True)

        elif "Tình trạng" in label:
            data["status"] = value_td.get_text(strip=True)


        elif "Ngành nghề chính" in label:

            a_tag = value_td.find("a")
            if a_tag:
                data["business"]["main"] = a_tag.get_text(strip=True)

            full_text = value_td.get_text(" ", strip=True)
            if "Chi tiết:" in full_text:
                data["business"]["detail"] = full_text.split("Chi tiết:")[-1].strip()

    return data


# ========================
# MAIN
# ========================
with open("companies_full.json", "r", encoding="utf-8") as f:
    companies = json.load(f)

results = []

for i, c in enumerate(companies, 1):
    if c.get("mst") and c.get("name"):
        print(f"\n------ {i} -------\n")

        data = parse_company(c["mst"], c["name"])

        print("URL:", data["url"])
        print("Tên:", data["name"])
        print("Ngành:", data["business"]["main"])

        results.append(data)

        # delay tránh block
        time.sleep(random.uniform(1, 3))


# ========================
# SAVE JSON
# ========================
with open("final_data.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

# 3. Phán đoán khả năng tuyển dụng 1 ngành nghề cụ thể


In [ ]:
import json

# ===== KEYWORDS =====
IT_KEYWORDS = {
    "công nghệ": 40,
    "phần mềm": 40,
    "hệ thống": 30,
    "dịch vụ": 20,
    "media": 20,
    "tư vấn": 20,
    "bán lẻ": 10,
    "bán buôn": 10,
}

LOGISTICS_KEYWORDS = {
    "vận tải": 50,
    "xuất khẩu": 40,
    "nhập khẩu": 40,
    "xuất nhập khẩu": 50,
    "giao nhận": 40,
    "hải quan": 40,
    "kho": 30,
}

NEGATIVE_IT = {
    "nhà hàng": -30,
    "ăn uống": -30,
    "sản xuất": -20,
    "may": -15,
}

# ===== SCORING FUNCTION =====
def calculate_score(text, keywords):
    score = 0
    matched = []

    for k, v in keywords.items():
        if k in text:
            score += v
            matched.append(k)

    return score, matched


def calculate_negative(text, keywords):
    score = 0
    matched = []

    for k, v in keywords.items():
        if k in text:
            score += v
            matched.append(k)

    return score, matched


def analyze_company(company):
    text = (
        (company.get("business", {}).get("main") or "") + " " +
        (company.get("business", {}).get("detail") or "")
    ).lower()

    it_score, it_matched = calculate_score(text, IT_KEYWORDS)
    log_score, log_matched = calculate_score(text, LOGISTICS_KEYWORDS)
    neg_score, neg_matched = calculate_negative(text, NEGATIVE_IT)

    it_score += neg_score

    # normalize
    it_score = max(0, min(it_score, 100))
    log_score = max(0, min(log_score, 100))

    return {
        "name": company["name"],
        "it_score": it_score,
        "logistics_score": log_score,
        "it_keywords": it_matched,
        "logistics_keywords": log_matched,
        "negative_keywords": neg_matched
    }


# ===== MAIN =====
def analyze_all(data):
    results = [analyze_company(c) for c in data]

    # sort theo IT score
    results.sort(key=lambda x: x["it_score"], reverse=True)

    return results


# ===== RUN =====
if __name__ == "__main__":
    with open("final_data.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    results = analyze_all(data)

    for r in results:
        print("=" * 50)
        print(f"🏢 {r['name']}")
        print(f"👉 IT score: {r['it_score']}")
        print(f"👉 Logistics score: {r['logistics_score']}")
        print(f"   + IT keywords: {r['it_keywords']}")
        print(f"   + Logistics keywords: {r['logistics_keywords']}")
        print(f"   - Negative: {r['negative_keywords']}")